<img src="../logo.png" alt="Vegeta" width="240">

# Race ingest — what the races ask of an aircraft: deep statistics of the laps, and the design factors for the raptors

The race runs of notebook 50 (`wf R1`–`R4`, the reference 5-inch quad in MuJoCo) digested: every lap's kinematics
second by second, the regimes it is flown in, the distributions behind the peaks, the braking and the corners and the
straights one by one, the power duty, the vertical content, the closest approaches — and from them the **design
factors**: what an aircraft built for these courses must do (hover thrust-to-weight, lateral g and turn radius at the
speeds used, body rates, cruise and top speed, the stall-speed ceiling and the share of a lap below wing-borne speed,
energy and power, footprint, braking, vertical speeds), per course and over all, with the margins stated. The factors
go to `designs/data/race_design_factors.json` for the raptor design notebook — **not connected yet**.

**Sources.** The notebook scans where the workflows and the CLI write their runs (`races.ingest.scan`), takes the
newest completed MuJoCo lap per course, and uses the mock flyer where no lap is on file (said so in every table: a mock
lap is the schedule's lower bound, flown without lag). Nothing is simulated here beyond the mock; `wf R1`–`R4` make the
real laps.

In [ ]:
DEBUG = True
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import os, sys, time, json
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
sys.path.insert(0, "../..")
from races import analysis, courses, ingest, metrics, mock, refquad, replay
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 120); pd.set_option("display.max_colwidth", 80)
from vegeta import yanus
GATE = yanus.enter("51_race_ingest")            # takes nothing it must have: it scans for runs and falls back to the mock
display(GATE.table())
T0 = time.time()

## 1. Scan: what is on disk, and where it was supposed to be

In [ ]:
SCAN = ingest.scan()
if len(SCAN):
    display(SCAN)
else:
    print("no race runs found; expected locations:")
for k, v in ingest.expected_locations().items():
    print(f"  {k:40s} {v}")
COURSES = {slug: fn() for slug, fn in courses.COURSES.items()}
LAPS, SOURCE = {}, {}
for slug, c in COURSES.items():
    cands = SCAN[(SCAN["course"] == slug) & (SCAN["state"] == "completed") & SCAN["trajectory"] & (SCAN["mode"] != "mock")] if len(SCAN) else SCAN
    if len(cands):
        run = Path("../..") / cands.iloc[-1]["run"]
        tr = run / "cases" / "race" / "trajectory.npz"
        LAPS[slug] = {"history": np.load(tr)["history"], "events": [tuple(e) for e in (json.loads((run / "cases" / "race" / "stats.json").read_text()).get("events") or [])], "run": str(run)}
        SOURCE[slug] = f"MuJoCo lap {cands.iloc[-1]['run']}"
    else:
        r = mock.run_mock(c)
        LAPS[slug] = {"history": r["history"], "events": r["events"], "run": None}
        SOURCE[slug] = "mock (no completed MuJoCo lap on file)"
display(pd.Series(SOURCE, name="source of the lap").to_frame())

## 2. The aircraft and the courses' own demands

The reference quad's envelope (what it could do) and what each course asks before any aircraft flies it: the
openings, the corners, the legs, the vertical, the schedule's speeds.

In [ ]:
display(pd.Series(metrics.envelope(), name="reference quad envelope").to_frame())
display(pd.DataFrame({c.name: analysis.course_demands(c) for c in COURSES.values()}).round(2))

## 3. Each lap, deep

For every course: the path coloured by speed and by load factor (plan and elevation), the timeline (speed, g, power,
tilt and rates with the zone passages and the collisions), the regimes, the percentiles behind the peaks, the braking
episodes, the corners, the straights, the vertical, the power duty, the closest approaches.

In [ ]:
REPORTS = {}
for slug, c in COURSES.items():
    lap = LAPS[slug]
    name = f"{c.name} ({'MuJoCo' if lap['run'] else 'mock'})"
    R = analysis.lap_report(lap["history"], c, lap["events"])          # the resets' teleports are blanked, not counted as manoeuvres
    REPORTS[name] = R
    k = R["kinematics"]
    display(Markdown(f"### {name}\n\n*{SOURCE[slug]}*"))
    fig, axes = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={"height_ratios": [2, 1]})
    analysis.plot_path_coloured(c, k, ax=axes[0], by="speed", view="plan"); analysis.plot_path_coloured(c, k, ax=axes[1], by="load_g", view="elevation"); plt.tight_layout(); plt.show()
    analysis.plot_timeline(k, lap["events"], title=name); plt.show()
    display(Markdown("**regimes**")); display(R["regimes"].round(3))
    display(Markdown("**percentiles while moving**")); display(R["percentiles"].round(2))
    display(Markdown(f"**braking episodes** ({len(R['braking'])}) and **acceleration episodes** ({len(R['accelerating'])})")); display(R["braking"].round(2).head(12))
    display(Markdown(f"**corners** ({len(R['corners'])})")); display(R["corners"].round(2).head(12))
    display(Markdown(f"**straights** ({len(R['straights'])})")); display(R["straights"].round(2).head(12))
    display(pd.Series(R["vertical"], name="vertical").to_frame().T.round(2)); display(pd.Series(R["power"], name="power duty").to_frame().T.round(2))
    display(Markdown("**closest approaches**")); display(R["closest approaches"].round(2))
print(f"§ 3 in {time.time() - T0:.0f} s")

## 4. Across the courses

The distributions side by side, and the corner demand against the thrust-vector envelope (lateral g against speed,
turn radius against speed, with the 45°, 60° and 70° tilt lines).

In [ ]:
analysis.plot_distributions(REPORTS); plt.show()
analysis.plot_corner_envelope(REPORTS); plt.show()
REG = pd.concat({n: r["regimes"]["time share"] for n, r in REPORTS.items()}, axis=1)
REG.T.plot.barh(stacked=True, figsize=(12, 3.5), title="time share by speed regime"); plt.grid(alpha=0.3, axis="x"); plt.show()

## 5. The design factors for the raptors

One column per course and the **all courses** column (the maximum of the demands, the minimum of the allowances, the
median of the descriptive rows). The margins turn a measured demand into a requirement: thrust-to-weight × 1.3,
lateral g × 1.2, body rates × 1.3, power × 1.2, energy × 1.25, speed × 1.1. `wing_borne_speed` is 12 m/s: below it a
winged raptor hovers or transitions; the stall-speed ceiling is the 10th percentile of the speed flown.

In [ ]:
FACT = analysis.design_factors(REPORTS)
display(FACT.round(2))
OUTF = analysis.factors_for_raptors(FACT)
for line in OUTF["reading"]:
    display(Markdown(f"- {line}"))
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, key in zip(ax, ("hover thrust / weight required", "lateral g required (p95 × margin)", "roll / pitch rate required (p99 × margin) [deg/s]", "share of time below wing-borne speed")):
    FACT.loc[key].drop("all courses").astype(float).plot.bar(ax=a, color="#3377dd", title=key); a.axhline(float(FACT.loc[key, "all courses"]), color="#d62828", ls="--", lw=1); a.grid(alpha=0.3, axis="y"); a.set_xlabel("")
plt.tight_layout(); plt.show()

## 6. Findings

- The **mock and the MuJoCo lap differ where it matters for design**: the real controller lags the racing line by
  metres in the technical sections, flies the hangar at half the mock's mean speed, and spends most of its time below
  wing-borne speed. The design factors are therefore taken from the real laps where they exist and from the mock
  only as a lower bound; the table says which.
- **What the courses ask**, in the all-courses column: a hover thrust-to-weight near 4, lateral 3 g, roll and pitch
  rates around 1000 °/s, a 1.2 m footprint (0.9 m in the hangar), cruise design speeds of 7–15 m/s with top speeds
  below 25 m/s, 1.5–2 Wh per lap. The reference quad's envelope exceeds every one of those by a factor of two or more
  except the body rates: these courses are flown far inside what a racing quad can do, and the limiting factor on
  lap time is the controller's tracking, not the airframe.
- **For a raptor**: the technical courses are rotorcraft work (hangar: 92 % of the time below 12 m/s), the open
  ones are wing work for half to three quarters of a lap. A winged raptor pays off on the farm, the quarry and the
  industrial zone and must still hover like a quad in every one of them; its span must pass a 1.2 m opening, or 0.9 m
  in the hangar; its energy advantage buys laps, since one 6S 1300 pack already covers several laps for the quad.
- **The raptor design notebook** will take `designs/data/race_design_factors.json` (not connected yet): the
  all-courses requirements, the per-course columns, the margins and the reading above.

In [ ]:
doc = {"schema": 1, "notebook": "51_race_ingest", "made_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()), "sources": SOURCE, "wing_borne_speed_m_s": 12.0, "race_laps": 3,
       "factors": OUTF, "course_demands": {c.name: analysis.course_demands(c) for c in COURSES.values()}, "regimes": {n: r["regimes"].to_dict() for n, r in REPORTS.items()},
       "percentiles": {n: r["percentiles"].to_dict() for n, r in REPORTS.items()}, "power": {n: r["power"] for n, r in REPORTS.items()}, "vertical": {n: {k: (list(v) if isinstance(v, tuple) else v) for k, v in r["vertical"].items()} for n, r in REPORTS.items()},
       "reference_quad": refquad.summary(), "envelope": metrics.envelope(), "scan": SCAN.to_dict(orient="records") if len(SCAN) else []}
def plain(x):
    if isinstance(x, dict): return {str(k): plain(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)): return [plain(v) for v in x]
    if isinstance(x, (np.floating, float)): return None if not np.isfinite(float(x)) else round(float(x), 4)
    if isinstance(x, (np.integer,)): return int(x)
    if isinstance(x, (np.bool_,)): return bool(x)
    return x
out = Path("../../designs/data/race_design_factors.json"); out.write_text(json.dumps(plain(doc), indent=1, ensure_ascii=False))
print("exported", out, f"{out.stat().st_size / 1e3:.0f} kB")
GATE.leave()
print(f"done in {time.time() - T0:.0f} s")